## API Client With Bearer Token

This notebook demonstrates the deployed authentication pattern for the API:
1. Sign in a user with Microsoft Entra ID via interactive auth code flow (browser pop-up).
2. Acquire a token for this API audience.
3. Call `/chat/stream` with `Authorization: Bearer <token>`.
4. The API performs server-side OBO to Fabric.

> **Prerequisite:** Add `http://localhost` as a redirect URI on your calling app registration (Entra ID > App registrations > Authentication > Add a platform > Mobile and desktop applications).


## Required Environment Variables

Set these before running the notebook:
- `CHAT_API_BASE_URL` (default: `http://localhost:8000`)
- `API_CLIENT_ID` (API app registration client ID)
- `CALLER_CLIENT_ID` (public client app registration client ID)
- `TENANT_ID` (optional, default: `common`)


In [ ]:
from __future__ import annotations

import os

import httpx
import msal

API_BASE_URL = os.getenv("CHAT_API_BASE_URL", "http://localhost:8000")
API_CLIENT_ID = os.getenv("API_CLIENT_ID", "")
CALLER_CLIENT_ID = os.getenv("CALLER_CLIENT_ID", "")
TENANT_ID = os.getenv("TENANT_ID", "common")

In [ ]:
def acquire_api_token() -> str:
    """Acquire a token for the API as a public client using interactive auth code flow."""
    if not API_CLIENT_ID:
        raise ValueError("Set API_CLIENT_ID to your API app registration's Client ID")
    if not CALLER_CLIENT_ID:
        raise ValueError("Set CALLER_CLIENT_ID to your public calling application's Client ID")

    app = msal.PublicClientApplication(
        CALLER_CLIENT_ID,
        authority=f"https://login.microsoftonline.com/{TENANT_ID}",
    )

    result = app.acquire_token_interactive(scopes=[f"api://{API_CLIENT_ID}/.default"])
    if "access_token" not in result:
        raise ValueError(
            f"Failed to acquire token: {result.get('error_description', result.get('error'))}"
        )

    return result["access_token"]


def stream_chat(api_token: str, prompt: str) -> None:
    """Send a streaming chat request to the API."""
    request_body = {"prompt": prompt}

    with httpx.stream(
        "POST",
        f"{API_BASE_URL}/chat/stream",
        json=request_body,
        headers={
            "Accept": "text/event-stream",
            "Authorization": f"Bearer {api_token}",
        },
        timeout=120,
    ) as response:
        response.raise_for_status()
        for line in response.iter_lines():
            if line:
                print(line)

In [ ]:
token = acquire_api_token()
stream_chat(token, "Summarize yesterday's sales by region")